# ML-06 — Signal Audit: Do the Flags Hold?

**Stretch task:** test whether simple, safe signals support the hypotheses behind the review flags. All conclusions are observational.

## 1. Distributions

The starter data contains heavy-tailed search/performance fields, so bucketed comparisons are more interpretable than relying on a mean alone.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
paths=[Path('data/raw/content_refresh_anonymized.csv'),Path('../../data/raw/content_refresh_anonymized.csv')]
p=next((x for x in paths if x.exists()),None)
if p is None: raise FileNotFoundError('Starter dataset not found.')
df=pd.read_csv(p)
df['declining']=(df['trend_direction'].astype(str).str.lower()=='down').astype(int)
print(df[['days_since_last_update','impressions_90d','avg_position','word_count']].describe(percentiles=[.5,.9,.95,.99]).round(2).to_string())

## 2. Signal tests #1 / #2 / #3

Each test compares observed decline rates across meaningful buckets. `CONFIRMED` means the directional hypothesis is supported in this snapshot; `OPPOSITE` means the data points the other way; `MIXED` means the comparison is not monotonic.

In [ ]:
def verdict(delta):
    if pd.isna(delta): return 'FALSE'
    if delta>0.02: return 'CONFIRMED'
    if delta<-0.02: return 'OPPOSITE'
    return 'MIXED'

# 1) Older pages
age_bins=[-np.inf,30,90,180,365,np.inf]
age_labels=['0-30','31-90','91-180','181-365','365+']
df['age_bucket']=pd.cut(df['days_since_last_update'],bins=age_bins,labels=age_labels,include_lowest=True)
age=df.groupby('age_bucket',observed=False)['declining'].agg(n='size',declining_rate='mean')
print('AGE SIGNAL'); print(age.round(3).to_string()); print('Verdict:',verdict(age['declining_rate'].get('181-365',np.nan)-age['declining_rate'].get('0-30',np.nan)))

# 2) Search visibility
vol_bins=[-np.inf,0,100,1000,3000,30000,np.inf]
vol_labels=['0','1-100','101-1k','1k-3k','3k-30k','30k+']
df['volume_bucket']=pd.cut(df['impressions_90d'],bins=vol_bins,labels=vol_labels,include_lowest=True)
vol=df.groupby('volume_bucket',observed=False)['declining'].agg(n='size',declining_rate='mean')
print('\nVISIBILITY SIGNAL'); print(vol.round(3).to_string()); print('Verdict:',verdict(vol['declining_rate'].get('3k-30k',np.nan)-vol['declining_rate'].get('1-100',np.nan)))

# 3) Average position
pos_bins=[-np.inf,3,10,20,50,np.inf]
pos_labels=['top-3','4-10','11-20','21-50','50+']
df['position_bucket']=pd.cut(df['avg_position'],bins=pos_bins,labels=pos_labels,include_lowest=True)
pos=df.groupby('position_bucket',observed=False)['declining'].agg(n='size',declining_rate='mean')
print('\nPOSITION SIGNAL'); print(pos.round(3).to_string()); print('Verdict:',verdict(pos['declining_rate'].get('50+',np.nan)-pos['declining_rate'].get('top-3',np.nan)))

## 3. Flag-linked test

The baseline flag uses **staleness + visibility**. I therefore test the combined condition directly and compare it with the overall observed decline rate. The rule is useful only as a prioritization heuristic; the test does not establish causality.

In [ ]:
flag=(df['days_since_last_update'].fillna(0)>=180)&(df['impressions_90d'].fillna(0)>=3000)
summary=pd.DataFrame({'group':['flagged','not_flagged'],'n':[int(flag.sum()),int((~flag).sum())],'declining_rate':[df.loc[flag,'declining'].mean(),df.loc[~flag,'declining'].mean()]})
print(summary.round(3).to_string(index=False))
print('Flag-linked verdict:',verdict(summary.loc[0,'declining_rate']-summary.loc[1,'declining_rate']))

## 4. What this means in practice

Use the signals as evidence for review prioritization, not as automatic content decisions. Staleness and visibility are reasonable first-pass filters, but mixed or weak signal results should remain visible rather than being hidden to make the rule look stronger.

## Self-check

- [x] Distributions are inspected.
- [x] Three safe signals have explicit verdicts.
- [x] The baseline's flag-linked assumption is tested directly.
- [x] Findings are observational and public-safe.